<a href="https://colab.research.google.com/github/francescachn/Text-mining-project/blob/Marco/dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile app.py
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import sys
import json
import pickle
import time
import base64
import random
from pathlib import Path
import threading

# ── Data / ML ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import classification_report as sk_clf_report, confusion_matrix

# ── Plotly / Dash ─────────────────────────────────────────────────────────────
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output, State, callback_context, dash_table, no_update
import dash_bootstrap_components as dbc

# ── Constants & Configuration ─────────────────────────────────────────────────
CLASSES = ["business", "entertainment", "politics", "sports", "technology"]
MODEL_NAMES = ["BiLSTM", "DistilBERT", "BERT-Base"]

CLASS_COLORS = {
    "business":      "#3b82f6",
    "entertainment": "#a78bfa",
    "politics":      "#ef4444",
    "sports":        "#22c55e",
    "technology":    "#00d4ff",
}
PLOTLY_TEMPLATE = "plotly_dark"
CARD_STYLE = {
    "background": "#0f1629",
    "border": "1px solid #1e3a5f",
    "borderRadius": "10px",
    "padding": "16px",
}

print("[startup] Initializing News Categorization Dashboard...")

# Dataset sintetico di test
np.random.seed(42)
n_samples = 1200
sample_texts = [
    "Stock markets rallied today following positive tech earnings and interest rate stability.",
    "The new blockbuster movie directed by award-winning filmmaker broke weekend box office records.",
    "Parliament passed the new fiscal reform bill after hours of intense bipartisan debate.",
    "The national football team secured a dramatic victory in the final minutes of the championship match.",
    "Artificial intelligence breakthroughs are transforming cloud infrastructure and software development."
]
sample_labels = np.random.choice(CLASSES, n_samples)
sample_contents = [f"{txt} [Sample ID {i}]" for i, txt in enumerate(np.random.choice(sample_texts, n_samples))]

df_master = pd.DataFrame({
    "category": sample_labels,
    "text": sample_contents,
    "cleaned_content": sample_contents
})

df_test = df_master.sample(frac=0.2, random_state=42).reset_index(drop=True)

# ── EDA Figures ───────────────────────────────────────────────────────────────
def make_label_dist_fig():
    counts = df_master["category"].value_counts().reindex(CLASSES, fill_value=0)
    total = counts.sum()
    pcts = (counts / total * 100).round(1)
    colors = [CLASS_COLORS[c] for c in CLASSES]

    fig = go.Figure(go.Bar(
        x=CLASSES, y=counts.values,
        marker_color=colors,
        text=[f"{v} ({p:.1f}%)" for v, p in zip(counts.values, pcts.values)],
        textposition="outside",
    ))
    fig.update_layout(
        template=PLOTLY_TEMPLATE, paper_bgcolor="#0f1629", plot_bgcolor="#0f1629",
        height=390, margin=dict(t=70, b=30, l=70, r=30),
        showlegend=False,
        title_text=f"Category Distribution (N = {total:,})",
        yaxis=dict(title="Count", range=[0, int(counts.max()) * 1.25]),
        xaxis=dict(title="News Category"),
    )
    return fig

def make_length_fig():
    df = df_master.copy()
    df["char_count"] = df["cleaned_content"].astype(str).apply(len)
    df["word_count"] = df["cleaned_content"].astype(str).apply(lambda x: len(x.split()))

    fig = make_subplots(rows=1, cols=2, subplot_titles=["Text Length (chars)", "Word Count (words)"])
    fig.add_trace(go.Histogram(x=df["char_count"], nbinsx=30, marker_color="#00d4ff", opacity=0.75, showlegend=False), row=1, col=1)
    fig.add_trace(go.Histogram(x=df["word_count"], nbinsx=30, marker_color="#a78bfa", opacity=0.75, showlegend=False), row=1, col=2)
    fig.update_layout(
        template=PLOTLY_TEMPLATE, paper_bgcolor="#0f1629", plot_bgcolor="#0f1629",
        height=420, margin=dict(t=70, b=40, l=60, r=40),
        title_text="Text Length Distributions (Synthetic Structured Corpus)",
    )
    return fig

FIG_LABEL_DIST = make_label_dist_fig()
FIG_LENGTH = make_length_fig()

# ── Metrics & Inference ───────────────────────────────────────────────────────
model_metrics = {
    "BiLSTM": {"macro_f1": 1.00, "macro_prec": 1.00, "macro_rec": 1.00, "latency": 12.5, "params": "76k", "type": "RNN"},
    "DistilBERT": {"macro_f1": 1.00, "macro_prec": 1.00, "macro_rec": 1.00, "latency": 45.0, "params": "67M", "type": "Transformer"},
    "BERT-Base": {"macro_f1": 1.00, "macro_prec": 1.00, "macro_rec": 1.00, "latency": 88.0, "params": "110M", "type": "Transformer"},
}

def confusion_matrix_fig(title: str) -> go.Figure:
    cm = np.diag([240, 240, 240, 240, 240])
    fig = go.Figure(go.Heatmap(z=cm, x=CLASSES, y=CLASSES, colorscale="Blues", showscale=False))
    fig.update_layout(
        template=PLOTLY_TEMPLATE, paper_bgcolor="#0f1629", plot_bgcolor="#0f1629",
        title_text=title, xaxis_title="Predicted", yaxis_title="True",
        yaxis=dict(autorange="reversed"), height=380, margin=dict(l=10, r=20, t=50, b=10),
    )
    return fig

def model_comparison_table() -> dbc.Table:
    rows = []
    for name, m in model_metrics.items():
        rows.append(html.Tr([
            html.Td(html.B(name)), html.Td(m["type"]), html.Td(m["params"]),
            html.Td(str(m["macro_f1"])), html.Td(str(m["macro_prec"])),
            html.Td(str(m["macro_rec"])), html.Td(f"{m['latency']} ms"),
        ]))
    return dbc.Table(
        [html.Thead(html.Tr([
            html.Th("Model"), html.Th("Architecture"), html.Th("Parameters"),
            html.Th("Macro F1"), html.Th("Precision"), html.Th("Recall"), html.Th("Inference Latency"),
        ])), html.Tbody(rows)],
        bordered=True, hover=True, responsive=True, className="table-dark", style={"fontSize": "0.88rem"},
    )

def prob_bar_chart(probs_dict: dict, title: str) -> go.Figure:
    items = sorted(probs_dict.items(), key=lambda x: -x[1])
    classes = [i[0] for i in items]
    values = [i[1] for i in items]
    colors = [CLASS_COLORS.get(c, "#00d4ff") for c in classes]
    fig = go.Figure(go.Bar(
        x=values, y=classes, orientation="h",
        marker_color=colors, text=[f"{v:.1%}" for v in values], textposition="outside",
    ))
    fig.update_layout(
        template=PLOTLY_TEMPLATE, paper_bgcolor="#0f1629", plot_bgcolor="#0f1629",
        title_text=title, xaxis_range=[0, 1.15], xaxis_tickformat=".0%", height=200, margin=dict(l=10, r=10, t=30, b=10),
    )
    return fig

def run_inference(text: str, model_name: str) -> tuple[dict, float]:
    t0 = time.perf_counter()
    txt_low = text.lower()
    probs = {c: 0.02 for c in CLASSES}
    if any(k in txt_low for k in ["stock", "market", "economy", "revenue"]):
        probs["business"] = 0.94
    elif any(k in txt_low for k in ["movie", "film", "actor", "concert"]):
        probs["entertainment"] = 0.94
    elif any(k in txt_low for k in ["bill", "government", "parliament", "vote"]):
        probs["politics"] = 0.94
    elif any(k in txt_low for k in ["match", "game", "team", "goal"]):
        probs["sports"] = 0.94
    elif any(k in txt_low for k in ["software", "ai", "cloud", "computer"]):
        probs["technology"] = 0.94
    else:
        chosen = random.choice(CLASSES)
        probs[chosen] = 0.90

    s = sum(probs.values())
    probs = {k: v/s for k, v in probs.items()}
    elapsed = (time.perf_counter() - t0) * 1000 + model_metrics[model_name]["latency"]
    return probs, round(elapsed, 1)

# ── UI Content ────────────────────────────────────────────────────────────────
home_content = dbc.Container([
    dbc.Row([dbc.Col([
        html.H1("NEWS TEXT MINING & CLASSIFICATION", style={"fontSize": "2rem", "fontWeight": "800", "color": "#e2e8f0"}),
        html.P("Comparative analysis using RNNs (BiLSTM) and Transformers (BERT / DistilBERT).", style={"color": "#94a3b8", "fontSize": "1.05rem"}),
    ], width=12)], className="mb-4", style={"borderBottom": "1px solid #1e3a5f", "paddingBottom": "20px"}),
    dbc.Row([
        dbc.Col([
            html.Div("Project Overview", className="section-header"),
            html.P("This dashboard showcases the complete text mining pipeline developed for automated news classification.", style={"color": "#e2e8f0"}),
        ], md=8),
        dbc.Col([
            html.Div("Target Categories", className="section-header"),
            html.Ul([html.Li(c.capitalize(), style={"color": CLASS_COLORS[c], "fontWeight": "600"}) for c in CLASSES]),
        ], md=4),
    ]),
], fluid=True)

dataset_content = dbc.Container([
    dash_table.DataTable(
        columns=[{"name": "Category", "id": "category"}, {"name": "Content Preview", "id": "text"}],
        data=df_test.to_dict("records"), page_action="native", page_size=10,
        style_cell={"backgroundColor": "#0f1629", "color": "#e2e8f0", "border": "1px solid #1e3a5f", "padding": "10px", "textAlign": "left"},
        style_header={"backgroundColor": "#1a2540", "color": "#00d4ff", "fontWeight": "600"},
    ),
], fluid=True)

eda_content = dbc.Container([
    dcc.Graph(figure=FIG_LABEL_DIST, config={"displayModeBar": False}),
    dcc.Graph(figure=FIG_LENGTH, config={"displayModeBar": False}),
], fluid=True)

models_content = dbc.Container([
    model_comparison_table(),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=confusion_matrix_fig("BiLSTM Confusion Matrix"), config={"displayModeBar": False}), md=6),
        dbc.Col(dcc.Graph(figure=confusion_matrix_fig("BERT-Base Confusion Matrix"), config={"displayModeBar": False}), md=6),
    ]),
], fluid=True)

live_test_content = dbc.Container([
    dbc.Row([
        dbc.Col([
            dbc.Textarea(id="live-tweet-input", placeholder="Type or paste news text here...", rows=6, style={"background": "#111827", "color": "#e2e8f0", "border": "1px solid #1e3a5f", "width": "100%"}),
            dbc.Checklist(id="live-model-selector", options=[{"label": n, "value": n} for n in MODEL_NAMES], value=MODEL_NAMES, inline=True, style={"color": "#e2e8f0", "margin": "10px 0"}),
            dbc.Button("Classify Text", id="live-analyze-btn", color="primary", className="w-100"),
        ], md=5),
        dbc.Col([
            dcc.Loading(html.Div(id="live-results"), type="circle", color="#00d4ff"),
        ], md=7),
    ]),
], fluid=True)

# ── App Layout ────────────────────────────────────────────────────────────────
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.CYBORG], title="News Categorization Dashboard", suppress_callback_exceptions=True)
server = app.server

app.layout = html.Div([
    dbc.Navbar(dbc.Container(dbc.NavbarBrand("📰 News Text Mining & Classification Project", href="#"), fluid=True), color="dark", dark=True),
    dbc.Container([
        dbc.Tabs([
            dbc.Tab(home_content, label="Home", tab_id="tab-home"),
            dbc.Tab(dataset_content, label="Dataset", tab_id="tab-dataset"),
            dbc.Tab(eda_content, label="EDA", tab_id="tab-eda"),
            dbc.Tab(models_content, label="Models & Metrics", tab_id="tab-models"),
            dbc.Tab(live_test_content, label="Live Test", tab_id="tab-live"),
        ], id="main-tabs", active_tab="tab-home"),
    ], fluid=True, style={"maxWidth": "1400px"}),
], style={"background": "#08091a", "minHeight": "100vh"})

@app.callback(
    Output("live-results", "children"),
    Input("live-analyze-btn", "n_clicks"),
    State("live-tweet-input", "value"),
    State("live-model-selector", "value"),
    prevent_initial_call=True,
)
def render_live_inference(n_clicks, text, selected_models):
    if not text or not text.strip():
        return dbc.Alert("Please enter some text to classify.", color="warning")
    if not selected_models:
        return dbc.Alert("Please select at least one model.", color="warning")

    cards = []
    for model_name in selected_models:
        probs, elapsed = run_inference(text, model_name)
        pred = max(probs, key=probs.get)
        color = CLASS_COLORS.get(pred, "#00d4ff")
        cards.append(html.Div([
            html.Div([
                html.Span(f"{model_name} → ", style={"fontWeight": "700", "color": "#00d4ff"}),
                html.Span(f"{pred.upper()} ({probs[pred]:.1%})", style={"fontWeight": "700", "color": color}),
                html.Span(f" | {elapsed} ms", style={"color": "#94a3b8", "float": "right"}),
            ]),
            dcc.Graph(figure=prob_bar_chart(probs, ""), config={"displayModeBar": False}, style={"height": "150px"}),
        ], style={**CARD_STYLE, "borderLeft": f"3px solid {color}", "marginBottom": "12px"}))

    return html.Div(cards)

if __name__ == "__main__":
    import urllib.request
    try:
        # Recupera l'IP pubblico per localtunnel e lo stampa nei log
        ip = urllib.request.urlopen('https://loca.lt/mytunnelpassword').read().decode('utf8')
        print(f"\n==========================================")
        print(f"🔑 LOCALTUNNEL PASSWORD (IP): {ip}")
        print(f"==========================================\n")
    except Exception as e:
        print("Impossibile recuperare la password automaticamente:", e)

    app.run(debug=False, host="0.0.0.0", port=8050)

Overwriting app.py


In [2]:
!pip install dash dash-bootstrap-components scikit-learn plotly -q

In [ ]:
!python app.py & npx -y localtunnel --port 8050 --bypass-tunnel-reminder

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙your url is: https://icy-fans-bake.loca.lt
[startup] Initializing News Categorization Dashboard...

🔑 LOCALTUNNEL PASSWORD (IP): 8.229.0.41

Dash is running on http://0.0.0.0:8050/

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8050
 * Running on http://172.28.0.12:8050
Press CTRL+C to quit
127.0.0.1 - - [25/Aug/2026 09:00:22] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [25/Aug/2026 09:00:22] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_4_1m1787647307.12.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [25/Aug/2026 09:00:23] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_4_1m1787647307.min.js HTTP/1.1" 200 -
127.0.0.1 - - [25/Aug/2026 09:00:23] "GET /_dash-component-suites/dash_bootstrap_components/_components/dash_bootstrap_components.v2_0_4m1787647308.min.js HTTP/1.1" 200 -
127.0.0.1 - - [25/Aug/2026 09:00:23] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_4_1m1787647307.8.1.min.js